# Golden Solution: Inventory Risk Assessment Benchmark

## Overview
This is the reference implementation for the Inventory Risk Assessment benchmark.
It demonstrates the correct way to:
- Load and validate inventory data
- Calculate risk scores
- Filter items deterministically
- Rank and export results

**Run each cell in order to see the complete solution.**

In [ ]:
# ============================================================================
# CELL 1: Import Libraries & Initialize
# ============================================================================

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

print("✓ Libraries imported")
print("✓ Starting Inventory Risk Assessment benchmark solution")
print()

In [ ]:
# ============================================================================
# CELL 2: Load & Validate Data
# ============================================================================

# Initialize debug checks
debug_checks = {
    'data_loaded': False,
    'required_columns_exist': False,
    'no_null_numeric_values': False,
    'risk_scores_positive': False
}

required_columns = [
    'item_id', 'category', 'stock_level', 'reorder_point', 'daily_demand',
    'demand_std_dev', 'lead_time_days', 'holding_cost_per_unit_day',
    'last_restock_date', 'stockout_count_last_month'
]

try:
    # Load the inventory data
    inventory_df = pd.read_csv('inventory_dataset.csv')
    debug_checks['data_loaded'] = True
    print(f"✓ Data loaded successfully")
    print(f"  - Total items: {len(inventory_df)}")
    print(f"  - Columns: {len(inventory_df.columns)}")
    
except FileNotFoundError:
    print("✗ Error: inventory_dataset.csv not found")
    raise

# Check all required columns exist
missing_columns = [col for col in required_columns if col not in inventory_df.columns]
if missing_columns:
    print(f"✗ Missing columns: {missing_columns}")
    raise ValueError(f"Missing columns: {missing_columns}")

debug_checks['required_columns_exist'] = True
print(f"✓ All {len(required_columns)} required columns present")

# Check for null values in numeric columns
numeric_cols = ['stock_level', 'reorder_point', 'daily_demand', 'demand_std_dev', 'lead_time_days']
null_count = inventory_df[numeric_cols].isnull().sum().sum()

if null_count > 0:
    print(f"✗ Found {null_count} null values in numeric columns")
    raise ValueError(f"Null values found in numeric columns")

debug_checks['no_null_numeric_values'] = True
print(f"✓ No null values in numeric columns")
print()

In [ ]:
# ============================================================================
# CELL 3: Calculate Safety Stock & Risk Scores
# ============================================================================

# Calculate safety stock baseline (protects against 99% of stockouts)
# Formula: (daily_demand × lead_time) + (2.33 × std_dev × √lead_time)

inventory_df['safety_stock'] = (
    (inventory_df['daily_demand'] * inventory_df['lead_time_days']) +
    (2.33 * inventory_df['demand_std_dev'] * np.sqrt(inventory_df['lead_time_days']))
).round(2)

print("✓ Safety stock calculated")

# Calculate days until stockout
inventory_df['days_until_stockout'] = np.maximum(
    1.0,  # Minimum 1 day (avoid division issues)
    inventory_df['stock_level'] / inventory_df['daily_demand']
).round(1)

# Safety stock violation check (1 if violated, 0 otherwise)
inventory_df['safety_violation'] = (
    inventory_df['stock_level'] < inventory_df['safety_stock']
).astype(int)

# Demand volatility (normalized)
inventory_df['demand_volatility'] = (
    inventory_df['demand_std_dev'] / inventory_df['daily_demand']
).round(2)

print("✓ Supporting metrics calculated")

# Calculate Risk Score
# Risk = (days_until_stockout × stockout_history) + (safety_violation × 10) + volatility

inventory_df['risk_score'] = (
    (inventory_df['days_until_stockout'] * inventory_df['stockout_count_last_month']) +
    (inventory_df['safety_violation'] * 10.0) +
    inventory_df['demand_volatility']
).round(2)

# Verify all risk scores are non-negative
min_risk = inventory_df['risk_score'].min()
max_risk = inventory_df['risk_score'].max()

if min_risk < 0:
    raise ValueError(f"Negative risk scores detected: minimum = {min_risk}")

debug_checks['risk_scores_positive'] = True

print(f"✓ Risk scores calculated")
print(f"  - Risk range: {min_risk:.2f} to {max_risk:.2f}")
print(f"  - Items at safety risk: {inventory_df['safety_violation'].sum()}")
print()

In [ ]:
# ============================================================================
# CELL 4: Filter & Rank Emergency Items
# ============================================================================

# Calculate 80th percentile risk threshold
percentile_80 = inventory_df['risk_score'].quantile(0.80)
print(f"✓ Risk threshold (80th percentile): {percentile_80:.2f}")

# Apply multi-condition filter
# Condition 1: risk_score >= 80th percentile
# Condition 2: stockout_count_last_month > 0 (proven problem history)
# Condition 3: stock_level < reorder_point (already below threshold)

critical_items = inventory_df[
    (inventory_df['risk_score'] >= percentile_80) &
    (inventory_df['stockout_count_last_month'] > 0) &
    (inventory_df['stock_level'] < inventory_df['reorder_point'])
].copy()

print(f"✓ Found {len(critical_items)} items meeting ALL criteria")

# Add urgency flags
critical_items['urgency_flag'] = critical_items.apply(
    lambda row: 'CRITICAL' if row['stock_level'] <= 0 else 'HIGH',
    axis=1
)

# Calculate recommended restock quantity
# = (reorder_point - current_stock) + (daily_demand × 7 days buffer)
critical_items['recommended_restock_quantity'] = (
    (critical_items['reorder_point'] - critical_items['stock_level']) +
    (critical_items['daily_demand'] * 7)
).round(0)

print(f"✓ Urgency flags & restock quantities calculated")

# Apply deterministic 3-key ranking
# 1. risk_score DESC (highest risk first)
# 2. stock_level ASC (lowest stock first)
# 3. item_id ASC (A-Z alphabetical)

emergency_restock_df = critical_items.sort_values(
    by=['risk_score', 'stock_level', 'item_id'],
    ascending=[False, True, True]
).head(10).copy()

# Extract item IDs as list[str]
emergency_item_ids = emergency_restock_df['item_id'].astype(str).tolist()

print(f"✓ Items ranked deterministically:")
print(f"  - Count: {len(emergency_item_ids)} items")
print(f"  - Top 3: {emergency_item_ids[:3]}")
print()

In [ ]:
# ============================================================================
# CELL 5: Export Results & Verification
# ============================================================================

# Verify all debug checks pass
print("✓ VERIFICATION CHECKPOINT:")
all_checks_pass = all(debug_checks.values())
for check_name, check_value in debug_checks.items():
    status = "✓" if check_value else "✗"
    print(f"  {status} {check_name}: {check_value}")

if not all_checks_pass:
    failed = [k for k, v in debug_checks.items() if not v]
    raise AssertionError(f"Failed checks: {failed}")

print(f"✓ All checks passed!")
print()

# Calculate summary metrics
total_emergency_stock_deficit = float(
    emergency_restock_df['recommended_restock_quantity'].sum()
)
total_emergency_stock_deficit = round(total_emergency_stock_deficit, 2)

highest_risk_score = float(
    emergency_restock_df['risk_score'].max() if len(emergency_restock_df) > 0 else 0.0
)
highest_risk_score = round(highest_risk_score, 2)

print(f"✓ Summary metrics:")
print(f"  - Total stock deficit: {total_emergency_stock_deficit} units")
print(f"  - Highest risk score: {highest_risk_score}")
print()

# Create output directory
os.makedirs('/content', exist_ok=True)

# Export CSV Audit Report
export_columns = [
    'item_id', 'category', 'stock_level', 'reorder_point',
    'risk_score', 'urgency_flag', 'days_until_stockout',
    'recommended_restock_quantity'
]

audit_df = emergency_restock_df[export_columns].copy()
csv_path = '/content/emergency_restock_audit.csv'
audit_df.to_csv(csv_path, index=False)
print(f"✓ Exported: {csv_path}")

# Export JSON Strategic Plan
strategic_plan = {
    'emergency_item_ids': emergency_item_ids,
    'total_emergency_stock_deficit': total_emergency_stock_deficit,
    'highest_risk_score': highest_risk_score,
    'debug_checks': debug_checks
}

json_path = '/content/emergency_action_plan.json'
with open(json_path, 'w') as f:
    json.dump(strategic_plan, f, indent=2)
print(f"✓ Exported: {json_path}")
print()

# Verify files exist
csv_exists = os.path.exists(csv_path)
json_exists = os.path.exists(json_path)

if not (csv_exists and json_exists):
    raise FileNotFoundError("Output files not created")

print(f"✓ Files verified to exist")
print()

# Display final output
print("=" * 70)
print("FINAL OUTPUT")
print("=" * 70)
print(json.dumps(strategic_plan, indent=2))
print("=" * 70)
print(f"✓ Solution complete!")